In [5]:
import os, warnings, numpy as np, pandas as pd, GPy

warnings.filterwarnings("ignore")
GPy.util.linalg.jitchol_MAXTRIES = 20

DATA_ROOT = r"C:/Users/sysdp/OneDrive - 島根大学/Desktop/Research/FPGA DATA/Raw-data_exahaustive"
OUT_ROOT = r"C:/Users/sysdp/OneDrive - 島根大学/Desktop/Research/MY PAPERS/Experiment 2/hybrid_al_sde"

MODE        = "all"   # "single" or "all"
SINGLE_FPGA = 1
SINGLE_PATH = 1
FPGA_START, FPGA_END, NUM_PATHS = 1, 50, 32

AL_RATIO         = 0.02
SDE_RATIO        = 0.08
K_NEIGHBORS      = 2
ROUGHNESS_THRESH = 2.0
SEED             = 42
MAX_ITERS = 200
CALIB_FPGA = 1


def load_dat(fpga, path):
    grid = np.loadtxt(os.path.join(DATA_ROOT, f"FPGA-{fpga}", f"data{path}.dat"), delimiter=",")
    rows, cols = grid.shape
    rec = [(x, y, grid[y, x]) for y in range(rows) for x in range(cols) if grid[y, x] != 0.0]
    df = pd.DataFrame(rec, columns=["X", "Y", "Value"])
    median, std = df.Value.median(), df.Value.std()
    return df[np.abs(df.Value - median) <= 3 * std].reset_index(drop=True)


def gpr(train_df, test_df):
    m = GPy.models.GPRegression(train_df[["X","Y"]].values, train_df[["Value"]].values,
                                 GPy.kern.RBF(input_dim=2), normalizer=True)
    m.optimize(max_iters=MAX_ITERS, messages=False)
    mu, var = m.predict(test_df[["X","Y"]].values)
    return mu.flatten(), var.flatten()


def active_learning(df):
    rng = np.random.RandomState(SEED)
    n, step = len(df), max(1, round(len(df) * 0.01))
    target = max(1, round(n * AL_RATIO))
    train = list(rng.choice(df.index, size=step, replace=False))
    while len(train) < target:
        rem = [i for i in df.index if i not in train]
        _, var = gpr(df.loc[train], df.loc[rem])
        pick = min(step, target - len(train), len(rem))
        train += [rem[i] for i in np.argsort(var)[:pick]]
    return train


def adaptive_sde(df, train):
    target = max(1, round(len(df) * SDE_RATIO))
    ab = {}
    for j in train:
        xj, yj = df.loc[j,"X"], df.loc[j,"Y"]
        others = sorted([p for p in train if p != j],
                        key=lambda p: (df.loc[p,"X"]-xj)**2+(df.loc[p,"Y"]-yj)**2)
        nn = others[:K_NEIGHBORS]
        if not nn: ab[j]=(1,1); continue
        dx = np.mean([abs(df.loc[p,"X"]-xj) for p in nn])
        dy = np.mean([abs(df.loc[p,"Y"]-yj) for p in nn])
        bump = 1 if np.std([df.loc[p,"Value"] for p in nn]) > ROUGHNESS_THRESH else 0
        ab[j] = (int(np.ceil(dx))+bump, int(np.ceil(dy))+bump)
    default_ab = (int(np.mean([v[0] for v in ab.values()])), int(np.mean([v[1] for v in ab.values()])))
    selected, sde = list(train), []
    for i in np.random.RandomState(SEED).permutation(df.index):
        if i in train or len(sde) >= target: continue
        x, y = df.loc[i,"X"], df.loc[i,"Y"]
        if all(abs(df.loc[j,"X"]-x) > ab.get(j,default_ab)[0] or
               abs(df.loc[j,"Y"]-y) > ab.get(j,default_ab)[1] for j in selected):
            sde.append(i); selected.append(i); ab[i] = default_ab
    if len(sde) < target:
        leftover = [i for i in np.random.RandomState(SEED).permutation(df.index)
                    if i not in train and i not in sde]
        sde += leftover[:target-len(sde)]
    return sde


calib_positions = {}


def get_calib_positions(path):
    if path not in calib_positions:
        df = load_dat(CALIB_FPGA, path)
        al = active_learning(df)
        sde = adaptive_sde(df, al)
        calib_positions[path] = set(map(tuple, df.loc[al + sde, ["X", "Y"]].values))
    return calib_positions[path]


def run_path(fpga, path):
    df = load_dat(fpga, path)
    pos_set = get_calib_positions(path)
    train_idx = df[df.apply(lambda r: (r.X, r.Y) in pos_set, axis=1)].index.tolist()
    test_idx = [i for i in df.index if i not in train_idx]
    mu, var = gpr(df.loc[train_idx], df.loc[test_idx])

    out = df.loc[test_idx, ["X", "Y", "Value"]].copy()
    out["Predicted"] = mu
    out["Variance"] = var

    out_dir = os.path.join(OUT_ROOT, f"FPGA{fpga}")
    os.makedirs(out_dir, exist_ok=True)
    out.to_csv(
        os.path.join(out_dir, f"fpga{fpga}_path{path}_hybrid_al_sde.csv"), index=False
    )
    print(f"FPGA-{fpga} Path-{path}: train={len(train_idx)}/{len(df)} saved.")


if __name__ == "__main__":
    jobs = [(SINGLE_FPGA, SINGLE_PATH)] if MODE == "single" else \
        [(f, p) for f in range(FPGA_START, FPGA_END+1) for p in range(1, NUM_PATHS+1)]
    for fpga, path in jobs:
        run_path(fpga, path)

FPGA-1 Path-1: train=315/3155 saved.
FPGA-1 Path-2: train=317/3173 saved.
FPGA-1 Path-3: train=317/3173 saved.
FPGA-1 Path-4: train=317/3173 saved.
FPGA-1 Path-5: train=317/3173 saved.
FPGA-1 Path-6: train=317/3173 saved.
FPGA-1 Path-7: train=317/3173 saved.
FPGA-1 Path-8: train=317/3173 saved.
FPGA-1 Path-9: train=317/3173 saved.
FPGA-1 Path-10: train=317/3173 saved.
FPGA-1 Path-11: train=317/3173 saved.
FPGA-1 Path-12: train=317/3173 saved.
FPGA-1 Path-13: train=317/3173 saved.
FPGA-1 Path-14: train=317/3173 saved.
FPGA-1 Path-15: train=317/3173 saved.
FPGA-1 Path-16: train=317/3173 saved.
FPGA-1 Path-17: train=317/3173 saved.
FPGA-1 Path-18: train=317/3173 saved.
FPGA-1 Path-19: train=317/3173 saved.
FPGA-1 Path-20: train=317/3173 saved.
FPGA-1 Path-21: train=317/3173 saved.
FPGA-1 Path-22: train=317/3173 saved.
FPGA-1 Path-23: train=317/3173 saved.
FPGA-1 Path-24: train=317/3173 saved.
FPGA-1 Path-25: train=317/3173 saved.
FPGA-1 Path-26: train=317/3173 saved.
FPGA-1 Path-27: train

In [10]:
import os, re, pandas as pd

OUT_ROOT = r"C:/Users/sysdp/OneDrive - 島根大学/Desktop/Research/MY PAPERS/Experiment 2/hybrid_al_sde"

dfs = []
for fpga_dir in os.listdir(OUT_ROOT):
    fpga_path = os.path.join(OUT_ROOT, fpga_dir)
    for f in os.listdir(fpga_path):
        if f.endswith("hybrid_al_sde.csv"):
            path = int(re.findall(r"path(\d+)", f)[0])
            fpga = int(re.findall(r"\d+", fpga_dir)[0])
            df = pd.read_csv(os.path.join(fpga_path, f))[
                ["X", "Y", "Value", "Predicted", "Variance"]
            ]
            df["FPGA"], df["PATH"] = int(fpga), int(path)
            dfs.append(df)

combined = pd.concat(dfs, ignore_index=True)
combined = combined.sort_values(["FPGA", "PATH"]).reset_index(drop=True)
combined.to_csv(f"{OUT_ROOT}/combined_hybrid_al_sde.csv", index=False)


In [11]:
import pandas as pd

OUT_ROOT = r"C:/Users/sysdp/OneDrive - 島根大学/Desktop/Research/MY PAPERS/Experiment 2/hybrid_al_sde"

df = pd.read_csv(f"{OUT_ROOT}/combined_hybrid_al_sde.csv")
df = df.sort_values(["FPGA", "PATH"]).reset_index(drop=True)
df.to_csv(f"{OUT_ROOT}/combined_hybrid_al_sde.csv", index=False)


In [12]:
import numpy as np, pandas as pd

ROOT_FOLDER = r"C:/Users/sysdp/OneDrive - 島根大学/Desktop/Research/FPGA DATA/Raw-data_exahaustive"
OUT_ROOT = r"C:/Users/sysdp/OneDrive - 島根大学/Desktop/Research/MY PAPERS/Experiment 2/hybrid_al_sde"
FPGA_START, FPGA_END = 1, 50
NUM_PATHS = 32


def load_dat(fp):
    grid = np.loadtxt(fp, delimiter=",")
    rows, cols = grid.shape
    vals = [grid[y, x] for y in range(rows) for x in range(cols) if grid[y, x] != 0.0]
    vals = pd.Series(vals)
    vals = vals[np.abs(vals - vals.median()) <= 3 * vals.std()]
    return vals


rows = []
for fpga in range(FPGA_START, FPGA_END + 1):
    for path in range(1, NUM_PATHS + 1):
        vals = load_dat(f"{ROOT_FOLDER}/FPGA-{fpga}/data{path}.dat")
        rows.append(
            {
                "FPGA": fpga,
                "PATH": path,
                "Max Value": vals.max(),
                "Min Value": vals.min(),
                "dspec": vals.max() - vals.min(),
            }
        )

dspec_df = pd.DataFrame(rows)

df = pd.read_csv(f"{OUT_ROOT}/combined_hybrid_al_sde.csv")
df = df.merge(dspec_df, on=["FPGA", "PATH"], how="left")
df.to_csv(f"{OUT_ROOT}/combined_hybrid_al_sde.csv", index=False)


In [13]:
import pandas as pd

OUT_ROOT = r"C:/Users/sysdp/OneDrive - 島根大学/Desktop/Research/MY PAPERS/Experiment 2/hybrid_al_sde"

df = pd.read_csv(f"{OUT_ROOT}/combined_hybrid_al_sde.csv")
df["delta"] = (df["Predicted"] - df["Value"]) / df["dspec"]

per_path = (
    df.groupby(["FPGA", "PATH"])["delta"].apply(lambda x: x.abs().mean()).reset_index()
)
per_path.columns = ["FPGA", "PATH", "Delta"]
per_path.to_csv(f"{OUT_ROOT}/hybrid_al_sde_delta_per_path.csv", index=False)

per_fpga = per_path.groupby("FPGA")["Delta"].mean().reset_index()
per_fpga.to_csv(f"{OUT_ROOT}/hybrid_al_sde_delta_per_fpga.csv", index=False)

overall = per_fpga["Delta"].mean()
pd.DataFrame({"Overall Delta": [overall]}).to_csv(
    f"{OUT_ROOT}/hybrid_al_sde_delta_overall.csv", index=False
)
print(f"Overall Delta: {overall:.6f}")


Overall Delta: 0.079898


In [4]:
import os, numpy as np, pandas as pd

DATA_ROOT = r"C:/Users/sysdp/OneDrive - 島根大学/Desktop/Research/FPGA DATA/Raw-data_exahaustive"
TEST_ROOT = r"C:/Users/sysdp/OneDrive - 島根大学/Desktop/Research/MY PAPERS/Experiment 2/hybrid_al_sde/FPGA1"
OUT_ROOT = r"C:/Users/sysdp/OneDrive - 島根大学/Desktop/Research/MY PAPERS/Experiment 2/hybrid_al_sde/FPGA1T"

FPGA = 1
NUM_PATHS = 32

os.makedirs(OUT_ROOT, exist_ok=True)

for path in range(1, NUM_PATHS + 1):
    # Load full raw data
    grid = np.loadtxt(
        os.path.join(DATA_ROOT, f"FPGA-{FPGA}", f"data{path}.dat"), delimiter=","
    )
    rows, cols = grid.shape
    rec = [
        (x, y, grid[y, x])
        for y in range(rows)
        for x in range(cols)
        if grid[y, x] != 0.0
    ]
    full = pd.DataFrame(rec, columns=["X", "Y", "Value"])
    median, std = full.Value.median(), full.Value.std()
    full = full[np.abs(full.Value - median) <= 3 * std].reset_index(drop=True)

    # Load test points CSV
    test_file = os.path.join(TEST_ROOT, f"fpga{FPGA}_path{path}_hybrid_al_sde.csv")
    if not os.path.exists(test_file):
        print(f"Path-{path}: test file not found, skipping.")
        continue
    test = pd.read_csv(test_file)[["X", "Y"]]

    # Training points = full data minus test points
    test_set = set(zip(test.X, test.Y))
    train = full[~full.apply(lambda r: (r.X, r.Y) in test_set, axis=1)]

    train.to_csv(
        os.path.join(OUT_ROOT, f"fpga{FPGA}_path{path}_train_hybrid_al_sde.csv"), index=False
    )
    print(f"Path-{path}: {len(train)} training points saved.")


Path-1: 315 training points saved.
Path-2: 317 training points saved.
Path-3: 317 training points saved.
Path-4: 317 training points saved.
Path-5: 317 training points saved.
Path-6: 317 training points saved.
Path-7: 317 training points saved.
Path-8: 317 training points saved.
Path-9: 317 training points saved.
Path-10: 317 training points saved.
Path-11: 317 training points saved.
Path-12: 317 training points saved.
Path-13: 317 training points saved.
Path-14: 317 training points saved.
Path-15: 317 training points saved.
Path-16: 317 training points saved.
Path-17: 317 training points saved.
Path-18: 317 training points saved.
Path-19: 317 training points saved.
Path-20: 317 training points saved.
Path-21: 317 training points saved.
Path-22: 317 training points saved.
Path-23: 317 training points saved.
Path-24: 317 training points saved.
Path-25: 317 training points saved.
Path-26: 317 training points saved.
Path-27: 317 training points saved.
Path-28: 317 training points saved.
P